# Projet Deep Learning — Reconnaissance d’expressions faciales avec FER2013

**Projet complet : parties 1 à 9 (socle + enrichissements/bonus)**  
**Dataset :** FER2013 — classification multiclasse de 7 expressions faciales.  
**Outils :** Python, TensorFlow/Keras, scikit-learn, pandas, NumPy, Matplotlib.

## 0. Environnement, imports et reproductibilité

Une graine fixe rend les initialisations, mélanges et échantillonnages aussi reproductibles que possible. Sur GPU, de faibles variations numériques peuvent néanmoins subsister selon la version de TensorFlow/CUDA.

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

SEED = 42

def reset_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

reset_seeds()

print("Python/TensorFlow prêt")
print("TensorFlow :", tf.__version__)
print("GPU(s) :", tf.config.list_physical_devices("GPU"))

## 1. Recherche, compréhension et préparation des données

### 1.1 Fiche descriptive de FER2013

- **Nom :** Facial Expression Recognition 2013 (FER2013).
- **Source de référence :** challenge *Challenges in Representation Learning — Facial Expression Recognition Challenge* ; copie publique Kaggle `msambare/fer2013`.
- **Auteurs / curateurs d'origine :** Pierre-Luc Carrier et Aaron Courville ; le dataset est décrit dans Goodfellow *et al.*, *Challenges in Representation Learning: A report on three machine learning contests*.
- **Liens :**
  - Kaggle : https://www.kaggle.com/datasets/msambare/fer2013
  - Article : https://arxiv.org/abs/1307.0414
  - Miroir technique CSV utilisé automatiquement si nécessaire : https://huggingface.co/datasets/chitradrishti/fer2013
- **Licence de la copie Kaggle :** *Database: Open Database, Contents: Database Contents*.
- **Nombre total standard :** 35 887 images.
- **Format :** images 48 × 48, niveaux de gris, pixels entiers entre 0 et 255.
- **Classes :** Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral.
- **Split officiel CSV :** `Training` / `PublicTest` / `PrivateTest`.

Le tableau calculé plus bas fournit le **nombre exact d'images par classe et par split** à partir du fichier effectivement chargé.

### 1.2 Téléchargement / localisation du dataset

Le notebook cherche d'abord `fer2013.csv`. S'il est absent, il tente de télécharger le miroir public CSV. En cas d'échec réseau, téléchargez manuellement FER2013 depuis Kaggle et placez `fer2013.csv` dans le répertoire du notebook.

In [ ]:
# Emplacements possibles (Colab puis répertoire courant)
candidates = [Path("/content/fer2013.csv"), Path.cwd() / "fer2013.csv"]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    target_dir = Path("/content") if Path("/content").exists() else Path.cwd()
    DATA_PATH = target_dir / "fer2013.csv"
    ZIP_PATH = target_dir / "fer2013.csv.zip"
    DATASET_URL = (
        "https://huggingface.co/datasets/chitradrishti/fer2013/"
        "resolve/main/fer2013.csv.zip?download=true"
    )

    print("fer2013.csv absent : téléchargement du miroir public...")
    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            csv_members = [m for m in z.namelist() if Path(m).name.lower() == "fer2013.csv"]
            if not csv_members:
                raise FileNotFoundError("fer2013.csv introuvable dans l'archive téléchargée.")
            member = csv_members[0]
            with z.open(member) as src, open(DATA_PATH, "wb") as dst:
                shutil.copyfileobj(src, dst)
        ZIP_PATH.unlink(missing_ok=True)
    except Exception as exc:
        raise RuntimeError(
            "Téléchargement automatique impossible. Téléchargez FER2013 depuis Kaggle "
            "et placez fer2013.csv dans /content ou le dossier courant."
        ) from exc

print("Dataset :", DATA_PATH)
print("Taille fichier (Mo) :", round(DATA_PATH.stat().st_size / 1024**2, 1))

### 1.3 Chargement, structure et contrôles d'intégrité

In [ ]:
df = pd.read_csv(DATA_PATH)

required_cols = {"emotion", "pixels"}
missing = required_cols - set(df.columns)
assert not missing, f"Colonnes manquantes : {missing}"

print("Shape du DataFrame :", df.shape)
print("Colonnes :", list(df.columns))
print("Valeurs manquantes :")
display(df.isna().sum().to_frame("n_missing"))

display(df.head(3))

# Vérifie la dimension d'une image
first_pixels = np.fromstring(df.loc[0, "pixels"], dtype=np.uint8, sep=" ")
print("Pixels par image :", first_pixels.size)
assert first_pixels.size == 48 * 48, "Le fichier ne correspond pas au format FER2013 48x48 attendu."

if len(df) != 35887:
    print(f"Attention : {len(df)} lignes au lieu des 35 887 attendues pour la version standard.")

### 1.4 Mapping des classes et distribution

Le problème est **multiclasse à 7 classes**. Les labels sont déjà des entiers 0–6 ; avec `sparse_categorical_crossentropy`, il n'est donc pas nécessaire de les convertir en one-hot vectors.

In [ ]:
CLASS_NAMES = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral",
}
N_CLASSES = len(CLASS_NAMES)
IMG_H = IMG_W = 48

class_counts = df["emotion"].value_counts().sort_index()
class_table = pd.DataFrame({
    "class_id": class_counts.index,
    "expression": [CLASS_NAMES[int(i)] for i in class_counts.index],
    "count": class_counts.values,
    "percent": 100 * class_counts.values / class_counts.sum(),
})
display(class_table)

plt.figure(figsize=(9, 4))
plt.bar(class_table["expression"], class_table["count"])
plt.ylabel("Nombre d'images")
plt.title("FER2013 — distribution des classes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

ratio = class_table["count"].max() / class_table["count"].min()
print(f"Rapport classe majoritaire / minoritaire : {ratio:.1f}x")
print("Classe majoritaire :", class_table.loc[class_table["count"].idxmax(), "expression"])
print("Classe minoritaire :", class_table.loc[class_table["count"].idxmin(), "expression"])

**Analyse du déséquilibre.** FER2013 est déséquilibré : `Happy` est fortement représentée alors que `Disgust` contient beaucoup moins d'exemples. Une accuracy globale peut donc masquer de mauvaises performances sur les petites classes. C'est pourquoi l'évaluation inclura aussi le rappel, la précision, le F1-score par classe et les moyennes macro.

### 1.5 Conversion des chaînes de pixels en tenseurs

In [ ]:
def parse_pixels(pixel_string, img_h=48, img_w=48):
    arr = np.fromstring(pixel_string, dtype=np.float32, sep=" ")
    if arr.size != img_h * img_w:
        raise ValueError(f"Taille inattendue : {arr.size} pixels")
    return arr.reshape(img_h, img_w, 1)

X_all = np.stack(df["pixels"].map(parse_pixels).to_numpy()).astype("float32")
y_all = df["emotion"].astype("int32").to_numpy()

print("X_all :", X_all.shape, X_all.dtype)
print("y_all :", y_all.shape, y_all.dtype)
print("Min / max avant normalisation :", X_all.min(), X_all.max())

### 1.6 Exemples d'images pour chaque catégorie

In [ ]:
def show_examples_per_class(X, y, class_names, n_per_class=5):
    classes = sorted(np.unique(y))
    fig, axes = plt.subplots(
        len(classes), n_per_class,
        figsize=(2 * n_per_class, 2 * len(classes)),
        squeeze=False
    )
    for row, cls in enumerate(classes):
        idx = np.where(y == cls)[0][:n_per_class]
        for col in range(n_per_class):
            ax = axes[row, col]
            if col < len(idx):
                ax.imshow(X[idx[col]].squeeze(), cmap="gray", vmin=0, vmax=255)
                ax.set_title(class_names[int(cls)])
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_examples_per_class(X_all, y_all, CLASS_NAMES, n_per_class=5)

### 1.7 Normalisation

Les intensités sont divisées par 255 afin de passer de `[0, 255]` à `[0, 1]`. Cette mise à l'échelle réduit l'amplitude numérique des entrées et rend l'optimisation par gradient plus stable. **Le même prétraitement devra être appliqué à toute image utilisée en inférence**, y compris plus tard aux visages détectés.

In [ ]:
X_all /= 255.0
print("Min / max après normalisation :", X_all.min(), X_all.max())

### 1.8 Constitution des ensembles train / validation / test

Lorsque la colonne `Usage` est disponible, on conserve le découpage officiel :

- `Training` → entraînement ;
- `PublicTest` → validation pour comparer les modèles et régler les hyperparamètres ;
- `PrivateTest` → **test final uniquement**.

Si `Usage` est absente, le notebook crée un split stratifié équivalent en principe.

In [ ]:
from sklearn.model_selection import train_test_split

if "Usage" in df.columns:
    usage = df["Usage"].astype(str).str.strip().to_numpy()
    train_mask = usage == "Training"
    val_mask = usage == "PublicTest"
    test_mask = usage == "PrivateTest"

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_val, y_val = X_all[val_mask], y_all[val_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]
    print("Split officiel FER2013 utilisé.")
else:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_all, y_all, test_size=0.10, random_state=SEED, stratify=y_all
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.111111, random_state=SEED, stratify=y_temp
    )
    print("Split stratifié manuel utilisé.")

print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

In [ ]:
def split_distribution(y, split_name):
    vc = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({
        "split": split_name,
        "class_id": vc.index,
        "expression": [CLASS_NAMES[int(i)] for i in vc.index],
        "count": vc.values,
        "percent": 100 * vc.values / vc.sum(),
    })

split_table = pd.concat([
    split_distribution(y_train, "train"),
    split_distribution(y_val, "validation"),
    split_distribution(y_test, "test"),
], ignore_index=True)
display(split_table)

# Le DataFrame brut n'est plus nécessaire pour l'entraînement.
del X_all

## 2. Modèle de référence — réseau dense

### 2.1 Principe

Architecture : `Image 48×48×1 → Flatten → Dense(256, ReLU) → Dense(7, Softmax)`.

- **Flatten** transforme les 2 304 pixels en un vecteur ; la structure spatiale n'est plus explicitement exploitée.
- Chaque neurone dense calcule une combinaison `z = w·x + b` : `w` représente les **poids** et `b` le **biais**.
- La **propagation avant** calcule les activations de couche en couche.
- **ReLU** `max(0, z)` introduit la non-linéarité.
- **Softmax** transforme les 7 logits en probabilités dont la somme vaut 1.
- La perte **sparse categorical cross-entropy** mesure l'écart entre la distribution prédite et le label entier correct.
- La **rétropropagation** calcule les gradients de la perte par rapport aux paramètres ; **Adam** met à jour les poids dans le sens qui réduit la perte.

Ce modèle sert de **baseline** : il permet de mesurer le gain apporté par l'exploitation spatiale des images via les convolutions.

In [ ]:
def build_baseline(input_shape=(48, 48, 1), n_classes=7):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ], name="baseline_dense")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

reset_seeds()
baseline = build_baseline((IMG_H, IMG_W, 1), N_CLASSES)
baseline.summary()

### 2.2 Entraînement du baseline

Un Early Stopping limite l'entraînement inutile et restaure les poids de l'époque présentant la meilleure `val_loss`.

In [ ]:
baseline_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    )
]

history_baseline = baseline.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=baseline_callbacks,
    verbose=1,
)

baseline_val_loss, baseline_val_acc = baseline.evaluate(X_val, y_val, verbose=0)
print(f"Baseline — val_loss={baseline_val_loss:.4f}, val_accuracy={baseline_val_acc:.4f}")

## 3. Construction du CNN avec Keras

### 3.1 Architecture choisie et justification

Architecture de référence :

`48×48×1 → Conv32 → Pool → Conv64 → Pool → Conv128 → Pool → Flatten → Dense128 → Dropout → Softmax7`

- **Filtres / feature maps :** les convolutions apprennent des détecteurs locaux (bords, textures, formes). Le nombre de feature maps augmente de 32 à 128 afin de représenter des motifs progressivement plus complexes.
- **Kernel 3×3 :** petit voisinage standard, efficace pour extraire des structures locales tout en limitant le nombre de paramètres.
- **Stride 1 + padding "same" :** la convolution conserve hauteur et largeur ; la réduction spatiale est donc contrôlée par le pooling.
- **ReLU :** activation simple et efficace qui évite de saturer comme des sigmoïdes sur une grande plage.
- **MaxPooling 2×2 :** divise approximativement hauteur et largeur par 2, réduit le coût et apporte une légère invariance aux petits déplacements.
- **Flatten + Dense(128) :** transforme les cartes de caractéristiques en représentation utilisée pour classifier l'expression.
- **Dropout 0,30 :** désactive aléatoirement 30 % des activations du Dense pendant l'entraînement pour limiter le surapprentissage.
- **Dense(7, Softmax) :** une probabilité par expression.

In [ ]:
def build_cnn(
    input_shape=(48, 48, 1),
    n_classes=7,
    dropout_rate=0.30,
    learning_rate=1e-3,
    augmentation=None,
    name="cnn_fer2013",
):
    inputs = keras.Input(shape=input_shape)
    x = inputs

    if augmentation is not None:
        x = augmentation(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

reset_seeds()
cnn = build_cnn(name="cnn_reference")
cnn.summary()

### 3.2 Dimensions et paramètres couche par couche

Avec `padding="same"`, les convolutions conservent les dimensions spatiales. Chaque MaxPooling 2×2 les divise par deux : **48 → 24 → 12 → 6**. Les canaux augmentent **1 → 32 → 64 → 128**. La couche Dense après `Flatten` concentre généralement la majorité des paramètres, car elle connecte les 4 608 valeurs aplaties aux 128 neurones.

In [ ]:
def model_layers_table(model):
    rows = []
    for layer in model.layers:
        try:
            out_shape = tuple(layer.output.shape)
        except Exception:
            out_shape = "?"
        rows.append({
            "Layer": layer.name,
            "Type": layer.__class__.__name__,
            "Output shape": out_shape,
            "Params": layer.count_params(),
        })
    return pd.DataFrame(rows)

layer_table = model_layers_table(cnn)
display(layer_table)
print("Nombre total de paramètres :", f"{cnn.count_params():,}".replace(",", " "))

largest = layer_table.loc[layer_table["Params"].idxmax()]
print(f"Couche la plus paramétrée : {largest['Layer']} ({largest['Params']:,} paramètres)")

## 4. Entraînement du CNN

### 4.1 Choix d'entraînement

- **Loss :** `sparse_categorical_crossentropy`, adaptée à 7 classes mutuellement exclusives avec labels entiers.
- **Optimiseur :** Adam, compromis robuste pour un premier CNN.
- **Learning rate initial :** `1e-3`.
- **Batch size :** 128, compromis entre stabilité du gradient, mémoire GPU et vitesse.
- **Époques maximales :** 40, mais Early Stopping peut interrompre avant.
- **Métrique :** accuracy, complétée ensuite par précision/rappel/F1 et matrice de confusion.
- **ReduceLROnPlateau :** réduit le learning rate si `val_loss` stagne.
- **ModelCheckpoint :** conserve le meilleur modèle selon `val_loss`.

In [ ]:
CHECKPOINT_DIR = Path("checkpoints_fer2013")
CHECKPOINT_DIR.mkdir(exist_ok=True)

EPOCHS = 40
BATCH_SIZE = 128

def make_callbacks(run_name, patience=6):
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            str(CHECKPOINT_DIR / f"{run_name}.keras"),
            monitor="val_loss", save_best_only=True
        ),
    ]

reset_seeds()
history_cnn = cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks("cnn_reference"),
    verbose=1,
)

cnn_val_loss, cnn_val_acc = cnn.evaluate(X_val, y_val, verbose=0)
print(f"CNN référence — val_loss={cnn_val_loss:.4f}, val_accuracy={cnn_val_acc:.4f}")

### 4.2 Courbes d'apprentissage et diagnostic du surapprentissage

In [ ]:
def plot_history(history, title="Historique"):
    hist = pd.DataFrame(history.history)
    epochs = np.arange(1, len(hist) + 1)

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, hist["loss"], label="train loss")
    plt.plot(epochs, hist["val_loss"], label="val loss")
    plt.xlabel("Époque")
    plt.ylabel("Loss")
    plt.title(title + " — Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, hist["accuracy"], label="train accuracy")
    plt.plot(epochs, hist["val_accuracy"], label="val accuracy")
    plt.xlabel("Époque")
    plt.ylabel("Accuracy")
    plt.title(title + " — Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()


def summarize_history(history):
    hist = pd.DataFrame(history.history)
    best_loss_idx = int(hist["val_loss"].idxmin())
    best_acc_idx = int(hist["val_accuracy"].idxmax())
    gap_at_best_loss = hist.loc[best_loss_idx, "accuracy"] - hist.loc[best_loss_idx, "val_accuracy"]

    print(f"Époques exécutées : {len(hist)}")
    print(f"Meilleure val_loss : {hist.loc[best_loss_idx, 'val_loss']:.4f} à l'époque {best_loss_idx + 1}")
    print(f"Meilleure val_accuracy observée : {hist.loc[best_acc_idx, 'val_accuracy']:.4f} à l'époque {best_acc_idx + 1}")
    print(f"Écart train-val accuracy à la meilleure val_loss : {gap_at_best_loss:.4f}")

    if gap_at_best_loss > 0.10:
        print("Diagnostic : écart important, signe de surapprentissage à surveiller.")
    elif gap_at_best_loss > 0.05:
        print("Diagnostic : surapprentissage modéré possible.")
    else:
        print("Diagnostic : écart train-validation limité au point retenu.")

    if len(hist) < EPOCHS:
        print("Early Stopping a interrompu l'entraînement avant la limite maximale et a restauré les meilleurs poids.")
    else:
        print("La limite maximale d'époques a été atteinte.")

plot_history(history_cnn, "CNN de référence")
summarize_history(history_cnn)

## 5. Évaluation et analyse des erreurs

### 5.1 Évaluation intermédiaire sur la **validation**

À ce stade, le modèle final n'a pas encore été choisi : pour respecter l'isolement du test, l'analyse détaillée ci-dessous porte sur la validation. Les mêmes outils seront réutilisés sur `PrivateTest` **une seule fois**, après les expériences de la partie 6.

In [ ]:
def evaluate_classifier(model, X, y, split_name="Validation"):
    loss, acc = model.evaluate(X, y, verbose=0)
    proba = model.predict(X, verbose=0)
    pred = np.argmax(proba, axis=1)

    report_dict = classification_report(
        y,
        pred,
        labels=list(range(N_CLASSES)),
        target_names=[CLASS_NAMES[i] for i in range(N_CLASSES)],
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).T
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))

    print(f"{split_name} — loss={loss:.4f}, accuracy={acc:.4f}")
    display(report_df.round(4))
    return {
        "loss": float(loss),
        "accuracy": float(acc),
        "proba": proba,
        "pred": pred,
        "report": report_df,
        "cm": cm,
    }

val_eval_ref = evaluate_classifier(cnn, X_val, y_val, "Validation — CNN référence")

### 5.2 Matrice de confusion et confusions dominantes

In [ ]:
def plot_confusion(cm, title, normalize=False):
    matrix = cm.astype("float64")
    if normalize:
        denom = matrix.sum(axis=1, keepdims=True)
        matrix = np.divide(matrix, denom, out=np.zeros_like(matrix), where=denom != 0)

    plt.figure(figsize=(8, 7))
    plt.imshow(matrix, cmap="Blues")
    plt.title(title)
    plt.xlabel("Classe prédite")
    plt.ylabel("Vraie classe")
    plt.xticks(range(N_CLASSES), [CLASS_NAMES[i] for i in range(N_CLASSES)], rotation=45)
    plt.yticks(range(N_CLASSES), [CLASS_NAMES[i] for i in range(N_CLASSES)])

    threshold = matrix.max() / 2 if matrix.size else 0
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            txt = f"{matrix[i, j]:.2f}" if normalize else str(int(matrix[i, j]))
            plt.text(j, i, txt, ha="center", va="center",
                     color="white" if matrix[i, j] > threshold else "black")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


def summarize_confusions(cm, top_k=6):
    row_sums = cm.sum(axis=1)
    recalls = np.divide(np.diag(cm), row_sums, out=np.zeros(N_CLASSES, dtype=float), where=row_sums != 0)
    order = np.argsort(recalls)[::-1]

    print("Rappels par classe :")
    for i in order:
        print(f"  {CLASS_NAMES[i]:>9s} : {recalls[i]:.3f}")

    off = cm.copy()
    np.fill_diagonal(off, 0)
    pairs = []
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            if i != j and off[i, j] > 0:
                pairs.append((off[i, j], i, j))
    pairs.sort(reverse=True)

    print("\nConfusions directionnelles les plus fréquentes :")
    for count, i, j in pairs[:top_k]:
        print(f"  {CLASS_NAMES[i]} → {CLASS_NAMES[j]} : {count} images")

plot_confusion(val_eval_ref["cm"], "Matrice de confusion — validation")
plot_confusion(val_eval_ref["cm"], "Matrice de confusion normalisée — validation", normalize=True)
summarize_confusions(val_eval_ref["cm"])

**Interprétation attendue.** Les classes avec le meilleur rappel sont celles dont la diagonale normalisée est la plus élevée. Les confusions `Fear/Sad/Neutral` ou `Angry/Sad` sont plausibles car les indices visuels peuvent être proches dans des images 48×48. `Disgust` est particulièrement difficile à apprendre en raison de son faible effectif. Les erreurs peuvent aussi provenir de poses non frontales, d'occlusions, d'un contraste faible, d'une expression ambiguë ou d'un label discutable.

### 5.3 Bonnes et mauvaises prédictions : image + vraie classe + classe prédite + probabilité

In [ ]:
def show_predictions(X, y_true, pred, proba, n=12, only_errors=False, seed=SEED):
    indices = np.where(y_true != pred)[0] if only_errors else np.arange(len(y_true))
    if len(indices) == 0:
        print("Aucun exemple disponible.")
        return

    rng = np.random.default_rng(seed)
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)

    cols = 4
    rows = int(np.ceil(len(chosen) / cols))
    plt.figure(figsize=(14, 3.5 * rows))

    for k, idx in enumerate(chosen, start=1):
        plt.subplot(rows, cols, k)
        plt.imshow(X[idx].squeeze(), cmap="gray", vmin=0, vmax=1)
        true_name = CLASS_NAMES[int(y_true[idx])]
        pred_name = CLASS_NAMES[int(pred[idx])]
        confidence = float(proba[idx, pred[idx]])
        plt.title(f"Vrai : {true_name}\nPrédit : {pred_name} ({confidence:.2f})")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

print("Exemples aléatoires :")
show_predictions(X_val, y_val, val_eval_ref["pred"], val_eval_ref["proba"], n=12, only_errors=False)

print("Exemples d'erreurs :")
show_predictions(X_val, y_val, val_eval_ref["pred"], val_eval_ref["proba"], n=16, only_errors=True)

In [ ]:
# Analyse quantitative de la confiance des erreurs
val_pred = val_eval_ref["pred"]
val_proba = val_eval_ref["proba"]
correct_mask = val_pred == y_val
confidence = val_proba[np.arange(len(y_val)), val_pred]

print(f"Confiance moyenne — bonnes prédictions : {confidence[correct_mask].mean():.3f}")
print(f"Confiance moyenne — erreurs            : {confidence[~correct_mask].mean():.3f}")

high_conf_errors = np.where((~correct_mask) & (confidence >= 0.80))[0]
print("Erreurs avec confiance ≥ 0.80 :", len(high_conf_errors))

if len(high_conf_errors):
    show_predictions(
        X_val, y_val, val_pred, val_proba,
        n=min(8, len(high_conf_errors)), only_errors=True, seed=7
    )

## 6. Expérimentations et amélioration — 3 expériences contrôlées

Le principe est de modifier **un seul facteur principal à la fois** par rapport au CNN de référence :

1. **Data Augmentation** : ajout de transformations légères ;
2. **Dropout** : passage de 0,30 à 0,50 ;
3. **Learning rate** : passage de `1e-3` à `3e-4`.

Toutes les comparaisons sont effectuées sur la **validation**. Le test reste intact jusqu'au choix final.

### 6.1 Infrastructure commune des expériences

In [ ]:
def train_model(model, run_name, epochs=40, batch_size=128):
    reset_seeds()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=make_callbacks(run_name),
        verbose=1,
    )
    return history


def result_row(name, modification, model, history):
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    hist = pd.DataFrame(history.history)
    return {
        "Expérience": name,
        "Modification": modification,
        "Val accuracy (poids restaurés)": float(val_acc),
        "Val loss (poids restaurés)": float(val_loss),
        "Meilleure val accuracy observée": float(hist["val_accuracy"].max()),
        "Meilleure val loss observée": float(hist["val_loss"].min()),
        "Époques": len(hist),
        "Paramètres": model.count_params(),
    }

### 6.2 Expérience 1 — Data Augmentation

**Hypothèse :** de petites transformations réalistes augmentent la diversité des exemples vus pendant l'entraînement et peuvent améliorer la généralisation sans modifier les labels.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
], name="data_augmentation")

exp1_model = build_cnn(
    dropout_rate=0.30,
    learning_rate=1e-3,
    augmentation=augmentation,
    name="exp1_augmentation",
)
exp1_history = train_model(exp1_model, "exp1_augmentation", epochs=40, batch_size=128)
exp1_val_loss, exp1_val_acc = exp1_model.evaluate(X_val, y_val, verbose=0)
print(f"Expérience 1 — val_loss={exp1_val_loss:.4f}, val_accuracy={exp1_val_acc:.4f}")

### 6.3 Expérience 2 — Dropout 0,50

**Hypothèse :** un Dropout plus fort réduit le surapprentissage. Si la régularisation devient trop forte, il peut toutefois provoquer du sous-apprentissage et diminuer la validation.

In [ ]:
exp2_model = build_cnn(
    dropout_rate=0.50,      # seule modification principale
    learning_rate=1e-3,
    augmentation=None,
    name="exp2_dropout_050",
)
exp2_history = train_model(exp2_model, "exp2_dropout_050", epochs=40, batch_size=128)
exp2_val_loss, exp2_val_acc = exp2_model.evaluate(X_val, y_val, verbose=0)
print(f"Expérience 2 — val_loss={exp2_val_loss:.4f}, val_accuracy={exp2_val_acc:.4f}")

### 6.4 Expérience 3 — Learning rate `3e-4`

**Hypothèse :** un pas d'apprentissage plus faible peut rendre les mises à jour plus stables et affiner la convergence, au prix d'un apprentissage potentiellement plus lent.

In [ ]:
exp3_model = build_cnn(
    dropout_rate=0.30,
    learning_rate=3e-4,     # seule modification principale
    augmentation=None,
    name="exp3_lr_3e4",
)
exp3_history = train_model(exp3_model, "exp3_lr_3e4", epochs=40, batch_size=128)
exp3_val_loss, exp3_val_acc = exp3_model.evaluate(X_val, y_val, verbose=0)
print(f"Expérience 3 — val_loss={exp3_val_loss:.4f}, val_accuracy={exp3_val_acc:.4f}")

### 6.5 Tableau comparatif et lecture critique

In [ ]:
experiment_rows = [
    result_row("Baseline dense", "Flatten + Dense, sans convolution", baseline, history_baseline),
    result_row("CNN référence", "3 blocs Conv/Pool, dropout 0.30, Adam 1e-3", cnn, history_cnn),
    result_row("Exp. 1 — Augmentation", "Flip horizontal + rotation 0.05 + zoom 0.08", exp1_model, exp1_history),
    result_row("Exp. 2 — Dropout 0.50", "Dropout 0.30 → 0.50", exp2_model, exp2_history),
    result_row("Exp. 3 — LR 3e-4", "Learning rate 1e-3 → 3e-4", exp3_model, exp3_history),
]
results_df = pd.DataFrame(experiment_rows)

ref_acc = float(results_df.loc[results_df["Expérience"] == "CNN référence", "Val accuracy (poids restaurés)"].iloc[0])

def observation(acc, name):
    if name == "Baseline dense":
        return "Point de comparaison : ne modélise pas explicitement la structure spatiale."
    if name == "CNN référence":
        return "Référence pour mesurer l'effet des expériences."
    delta = acc - ref_acc
    if delta >= 0.005:
        return f"Amélioration nette vs CNN référence : {delta:+.3f}."
    if delta <= -0.005:
        return f"Dégradation vs CNN référence : {delta:+.3f}."
    return f"Écart faible vs CNN référence : {delta:+.3f}."

results_df["Observation"] = [
    observation(acc, name)
    for acc, name in zip(results_df["Val accuracy (poids restaurés)"], results_df["Expérience"])
]

display(
    results_df.sort_values("Val accuracy (poids restaurés)", ascending=False)
              .reset_index(drop=True)
              .style.format({
                  "Val accuracy (poids restaurés)": "{:.4f}",
                  "Val loss (poids restaurés)": "{:.4f}",
                  "Meilleure val accuracy observée": "{:.4f}",
                  "Meilleure val loss observée": "{:.4f}",
              })
)

In [ ]:
# Comparaison visuelle des courbes de validation
histories = {
    "CNN référence": history_cnn,
    "Exp1 augmentation": exp1_history,
    "Exp2 dropout 0.50": exp2_history,
    "Exp3 LR 3e-4": exp3_history,
}

plt.figure(figsize=(8, 5))
for name, hist_obj in histories.items():
    vals = hist_obj.history["val_accuracy"]
    plt.plot(np.arange(1, len(vals) + 1), vals, label=name)
plt.xlabel("Époque")
plt.ylabel("Validation accuracy")
plt.title("Comparaison des expériences — validation accuracy")
plt.legend()
plt.tight_layout()
plt.show()

### 6.6 Choix du modèle final — uniquement à partir de la validation

Le modèle final est le CNN présentant la meilleure accuracy de validation **après restauration des meilleurs poids**. Le baseline dense n'est pas candidat au modèle final : il sert uniquement de référence pédagogique.

In [ ]:
models_by_name = {
    "CNN référence": cnn,
    "Exp. 1 — Augmentation": exp1_model,
    "Exp. 2 — Dropout 0.50": exp2_model,
    "Exp. 3 — LR 3e-4": exp3_model,
}

cnn_results = results_df[results_df["Expérience"].isin(models_by_name)].copy()
best_row = cnn_results.sort_values(
    ["Val accuracy (poids restaurés)", "Val loss (poids restaurés)"],
    ascending=[False, True]
).iloc[0]

best_name = best_row["Expérience"]
final_model = models_by_name[best_name]

print("Modèle final retenu :", best_name)
print(f"Validation accuracy : {best_row['Val accuracy (poids restaurés)']:.4f}")
print(f"Validation loss     : {best_row['Val loss (poids restaurés)']:.4f}")
print("Critère : meilleure validation accuracy parmi les CNN, sans consulter le test.")

### 6.7 Évaluation finale sur `PrivateTest`

**C'est la première utilisation du jeu de test pour comparer la performance finale.** Aucune décision d'architecture ou d'hyperparamètre ne doit être prise à partir des résultats ci-dessous.

In [ ]:
test_eval = evaluate_classifier(final_model, X_test, y_test, "TEST final — modèle retenu")

plot_confusion(test_eval["cm"], "Matrice de confusion — TEST final")
plot_confusion(test_eval["cm"], "Matrice de confusion normalisée — TEST final", normalize=True)
summarize_confusions(test_eval["cm"])

In [ ]:
print("Prédictions correctes et incorrectes sur le test :")
show_predictions(X_test, y_test, test_eval["pred"], test_eval["proba"], n=12, only_errors=False, seed=42)

print("Erreurs sur le test :")
show_predictions(X_test, y_test, test_eval["pred"], test_eval["proba"], n=16, only_errors=True, seed=123)

### 6.8 Synthèse automatique du modèle final

In [ ]:
report = test_eval["report"]
class_report = report.loc[[CLASS_NAMES[i] for i in range(N_CLASSES)]].copy()

best_recall_class = class_report["recall"].idxmax()
worst_recall_class = class_report["recall"].idxmin()
macro_f1 = float(report.loc["macro avg", "f1-score"])
weighted_f1 = float(report.loc["weighted avg", "f1-score"])

summary_table = pd.DataFrame({
    "Indicateur": [
        "Modèle final",
        "Validation accuracy",
        "Test accuracy",
        "Macro F1 test",
        "Weighted F1 test",
        "Meilleur rappel",
        "Rappel le plus faible",
    ],
    "Valeur": [
        best_name,
        f"{best_row['Val accuracy (poids restaurés)']:.4f}",
        f"{test_eval['accuracy']:.4f}",
        f"{macro_f1:.4f}",
        f"{weighted_f1:.4f}",
        f"{best_recall_class} ({class_report.loc[best_recall_class, 'recall']:.4f})",
        f"{worst_recall_class} ({class_report.loc[worst_recall_class, 'recall']:.4f})",
    ]
})
display(summary_table)

print("\nAnalyse critique :")
print("- Le CNN doit être comparé au baseline pour quantifier l'intérêt des convolutions.")
print("- La moyenne macro F1 est importante car FER2013 est déséquilibré.")
print("- Les classes à faible rappel et les plus fortes cases hors diagonale de la matrice de confusion indiquent les priorités d'amélioration.")
print("- Les limites intrinsèques restent la faible résolution 48x48, les expressions ambiguës, les poses/occlusions et le bruit d'annotation.")

### 6.9 Sauvegarde des livrables du socle

In [ ]:
FINAL_MODEL_PATH = Path("fer2013_final_model.keras")
RESULTS_PATH = Path("fer2013_experiments.csv")
CLASSES_PATH = Path("fer2013_class_names.json")

final_model.save(FINAL_MODEL_PATH)
results_df.to_csv(RESULTS_PATH, index=False)
with open(CLASSES_PATH, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print("Modèle sauvegardé :", FINAL_MODEL_PATH.resolve())
print("Tableau expériences :", RESULTS_PATH.resolve())
print("Mapping classes      :", CLASSES_PATH.resolve())

# Conclusion — socle obligatoire (Parties 1 à 6)

Ce notebook fournit le pipeline complet demandé pour le socle :

1. **FER2013 étudié et préparé** : structure, classes, déséquilibre, visualisation, normalisation et séparation train/validation/test ;
2. **baseline dense** entraîné comme point de comparaison ;
3. **CNN personnel** à trois blocs Conv/Pool avec dimensions et paramètres explicités ;
4. **entraînement contrôlé** par Early Stopping, réduction du learning rate et suivi des courbes ;
5. **évaluation multiclasse** : accuracy, précision, rappel, F1, matrices de confusion, exemples de prédictions et analyse des erreurs ;
6. **trois expériences** comparées sur validation, sélection automatique du meilleur CNN puis évaluation finale unique sur le test.

La prochaine étape sera le **bonus** : enrichissement (partie 7), détection de plusieurs visages / YOLO (partie 8), puis éventuellement vidéo (partie 9).

## Checklist du socle

- [x] Source, auteurs/curateurs et licence documentés
- [x] Nombre d'images, dimensions, format et classes analysés
- [x] Distribution des classes et déséquilibre visualisés
- [x] Exemples de chaque classe affichés
- [x] Normalisation et labels expliqués
- [x] Train / validation / test séparés ; test réservé à la fin
- [x] Baseline dense construit et entraîné
- [x] CNN construit et justifié
- [x] Dimensions et paramètres couche par couche
- [x] Courbes loss / accuracy et diagnostic de surapprentissage
- [x] Matrice de confusion et métriques par classe
- [x] Bonnes et mauvaises prédictions affichées avec probabilité
- [x] Trois expériences contrôlées
- [x] Tableau comparatif automatique
- [x] Choix du modèle final sur validation
- [x] Évaluation finale unique sur test
- [x] Sauvegarde du modèle final et des résultats

# 7. Enrichissement — Transfer Learning et fine-tuning

Le sujet propose, parmi les enrichissements, la **Data Augmentation**, le **Transfer Learning**, les réseaux pré-entraînés et la régularisation. La Data Augmentation et le Dropout ont déjà été étudiés dans la partie 6 ; on ajoute ici un véritable **Transfer Learning** avec **MobileNetV2 pré-entraîné sur ImageNet**.

### Principe

FER2013 contient des images **48×48 en niveaux de gris**, alors que MobileNetV2 attend une image RGB. Le pipeline d'entrée du modèle effectue donc :

`48×48×1 → redimensionnement 96×96 → duplication en 3 canaux → mise à l'échelle [-1,1] → MobileNetV2 → GlobalAveragePooling → Dropout → Softmax(7)`

On entraîne d'abord uniquement la tête de classification avec le backbone gelé, puis on réalise un **fine-tuning léger** des dernières couches avec un learning rate très faible. L'objectif est d'étudier si des caractéristiques visuelles apprises sur un grand dataset peuvent être transférées vers la reconnaissance d'expressions.

In [ ]:
# Partie 7 — Transfer Learning MobileNetV2
TRANSFER_AVAILABLE = True
TRANSFER_IMAGE_SIZE = 96
TRANSFER_EPOCHS_HEAD = 8
TRANSFER_EPOCHS_FINETUNE = 5


def build_transfer_model(input_shape=(48, 48, 1), n_classes=7, image_size=96):
    inputs = keras.Input(shape=input_shape, name="fer_input")

    # Augmentation modérée : les transformations restent plausibles pour un visage.
    x = layers.RandomFlip("horizontal", name="tl_flip")(inputs)
    x = layers.RandomRotation(0.04, name="tl_rotation")(x)
    x = layers.RandomZoom(0.06, name="tl_zoom")(x)

    # MobileNetV2 attend trois canaux et une taille >= 32x32.
    x = layers.Resizing(image_size, image_size, name="tl_resize")(x)
    x = layers.Concatenate(axis=-1, name="gray_to_rgb")([x, x, x])
    x = layers.Rescaling(scale=2.0, offset=-1.0, name="mobilenet_scaling")(x)

    backbone = keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(image_size, image_size, 3),
    )
    backbone.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(0.35, name="tl_dropout")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="tl_softmax")(x)

    model = keras.Model(inputs, outputs, name="fer2013_mobilenetv2_transfer")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, backbone


try:
    transfer_model, transfer_backbone = build_transfer_model(
        input_shape=(IMG_H, IMG_W, 1),
        n_classes=N_CLASSES,
        image_size=TRANSFER_IMAGE_SIZE,
    )
    transfer_model.summary()
except Exception as exc:
    TRANSFER_AVAILABLE = False
    transfer_model = None
    transfer_backbone = None
    print("Transfer Learning indisponible (souvent dû à l'absence d'accès réseau pour les poids ImageNet).")
    print("Détail :", exc)

### 7.1 Entraînement de la tête de classification

Le backbone est **gelé** : ses poids pré-entraînés ne sont pas modifiés. Seule la nouvelle tête de classification apprend FER2013. L'Early Stopping limite le surapprentissage et restaure les meilleurs poids selon `val_loss`.

In [ ]:
if TRANSFER_AVAILABLE:
    reset_seeds()
    transfer_head_callbacks = [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
    ]

    transfer_history_head = transfer_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=TRANSFER_EPOCHS_HEAD,
        batch_size=128,
        callbacks=transfer_head_callbacks,
        verbose=1,
    )

    plot_history(transfer_history_head, "Transfer Learning — tête de classification")
    transfer_val_head = evaluate_classifier(
        transfer_model, X_val, y_val, "Validation — Transfer Learning (backbone gelé)"
    )
else:
    print("Étape ignorée : poids pré-entraînés non disponibles.")

### 7.2 Fine-tuning contrôlé

On dégèle uniquement les **20 dernières couches** du backbone. Les couches de Batch Normalization restent gelées, ce qui évite de perturber fortement leurs statistiques avec un dataset différent. Le learning rate passe à `1e-5`, beaucoup plus faible que lors de l'apprentissage de la tête.

In [ ]:
if TRANSFER_AVAILABLE:
    transfer_backbone.trainable = True

    for layer in transfer_backbone.layers[:-20]:
        layer.trainable = False
    for layer in transfer_backbone.layers[-20:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    transfer_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    finetune_callbacks = [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        )
    ]

    transfer_history_finetune = transfer_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=TRANSFER_EPOCHS_FINETUNE,
        batch_size=128,
        callbacks=finetune_callbacks,
        verbose=1,
    )

    plot_history(transfer_history_finetune, "Fine-tuning — 20 dernières couches")
    transfer_val_finetune = evaluate_classifier(
        transfer_model, X_val, y_val, "Validation — Transfer Learning après fine-tuning"
    )
else:
    print("Fine-tuning ignoré : Transfer Learning non disponible.")

### 7.3 Comparaison avec le CNN retenu en partie 6

Cette comparaison utilise **la validation uniquement**. Le modèle de déploiement de la partie 6 n'est pas changé automatiquement après consultation du test : cela évite de transformer le jeu de test en jeu de sélection implicite.

In [ ]:
cnn_val_for_bonus = evaluate_classifier(
    final_model, X_val, y_val, "Validation — CNN retenu en partie 6"
)

bonus_rows = [{
    "Modèle": "CNN retenu — partie 6",
    "Val accuracy": cnn_val_for_bonus["accuracy"],
    "Val macro-F1": cnn_val_for_bonus["report"].loc["macro avg", "f1-score"],
}]

if TRANSFER_AVAILABLE:
    bonus_rows.append({
        "Modèle": "MobileNetV2 + fine-tuning",
        "Val accuracy": transfer_val_finetune["accuracy"],
        "Val macro-F1": transfer_val_finetune["report"].loc["macro avg", "f1-score"],
    })

bonus_comparison = pd.DataFrame(bonus_rows).sort_values("Val accuracy", ascending=False)
display(bonus_comparison.style.format({"Val accuracy": "{:.4f}", "Val macro-F1": "{:.4f}"}))

if TRANSFER_AVAILABLE:
    transfer_model.save("fer2013_transfer_mobilenetv2.keras")
    print("Modèle Transfer Learning sauvegardé : fer2013_transfer_mobilenetv2.keras")

# 8. Extension — Détection et analyse de plusieurs visages avec YOLO

Dans les parties 1 à 7, l'entrée est déjà un **visage recadré**. Ici, on passe à une chaîne de détection complète :

**Image → YOLO-Face → bounding boxes → extraction de chaque visage → prétraitement FER2013 → CNN → expression + probabilité**

Un modèle YOLO standard entraîné sur COCO détecte notamment la classe *person*, mais pas une classe *face* dédiée. On utilise donc **YOLOv8n-Face**, un détecteur spécialisé entraîné sur **WIDERFace**, puis le classifieur FER2013 déjà entraîné pour attribuer une expression à chaque visage.

Notions à distinguer :

- **classification** : une classe pour une image déjà recadrée ;
- **détection** : localisation par `bounding box` + score de confiance ;
- **IoU** : recouvrement entre deux bounding boxes ;
- **NMS** : suppression des boîtes redondantes se recouvrant fortement ;
- **precision / recall / mAP** : métriques adaptées à l'évaluation d'un détecteur lorsque des bounding boxes de vérité terrain sont disponibles.

FER2013 ne fournit pas de vérité terrain de détection multi-visages ; il sert ici à **classifier les crops détectés**, pas à calculer le mAP du détecteur.

In [ ]:
# Installation de l'implémentation YOLO si nécessaire.
import importlib.util
import hashlib
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import cv2
from ultralytics import YOLO

# Le dépôt Lindevs publie un poids YOLOv8n-Face compact (~6 MB).
YOLO_FACE_URL = (
    "https://github.com/lindevs/yolov8-face/releases/latest/download/"
    "yolov8n-face-lindevs.pt"
)
YOLO_FACE_PATH = Path("yolov8n-face-lindevs.pt")
YOLO_FACE_SHA256 = "b038ca653b503453a94f6e12d76feca6840b2a97d7a1322b4498c5e922f29832"

if not YOLO_FACE_PATH.exists():
    print("Téléchargement de YOLOv8n-Face...")
    urllib.request.urlretrieve(YOLO_FACE_URL, YOLO_FACE_PATH)

sha256 = hashlib.sha256(YOLO_FACE_PATH.read_bytes()).hexdigest()
if sha256 != YOLO_FACE_SHA256:
    raise RuntimeError(
        "Le SHA-256 du poids YOLO téléchargé ne correspond pas à celui publié par la source. "
        f"Obtenu={sha256}"
    )

face_detector = YOLO(str(YOLO_FACE_PATH))
print("Détecteur YOLO-Face chargé :", YOLO_FACE_PATH)

### 8.1 Fonctions de détection, prétraitement et classification

La cohérence du prétraitement est essentielle : chaque visage détecté est converti en niveaux de gris, redimensionné en **48×48**, normalisé dans `[0,1]`, puis fourni au même classifieur que pour FER2013.

In [ ]:
CLASS_NAMES_FR = {
    0: "Colere",
    1: "Degout",
    2: "Peur",
    3: "Joie",
    4: "Tristesse",
    5: "Surprise",
    6: "Neutre",
}


def box_iou_xyxy(box_a, box_b):
    """IoU de deux boîtes [x1, y1, x2, y2]."""
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union


def detect_faces_yolo(image_bgr, conf=0.35, imgsz=640):
    """Retourne [(xyxy, confidence), ...] pour tous les visages détectés."""
    result = face_detector.predict(
        source=image_bgr,
        conf=conf,
        imgsz=imgsz,
        verbose=False,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        return []

    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy()
    return [(box.astype(float), float(score)) for box, score in zip(xyxy, scores)]


def crop_face_with_margin(image_bgr, box_xyxy, margin=0.12):
    """Recadre un visage et ajoute une petite marge autour de la boîte YOLO."""
    h, w = image_bgr.shape[:2]
    x1, y1, x2, y2 = map(float, box_xyxy)
    bw, bh = x2 - x1, y2 - y1

    x1 = max(0, int(x1 - margin * bw))
    y1 = max(0, int(y1 - margin * bh))
    x2 = min(w, int(x2 + margin * bw))
    y2 = min(h, int(y2 + margin * bh))

    if x2 <= x1 or y2 <= y1:
        return None, (x1, y1, x2, y2)
    return image_bgr[y1:y2, x1:x2], (x1, y1, x2, y2)


def classify_face_crop(face_bgr, classifier=None):
    """Applique exactement le prétraitement FER2013 puis Softmax(7)."""
    if classifier is None:
        classifier = final_model

    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    x = resized.astype(np.float32) / 255.0
    x = x[None, ..., None]

    proba = classifier.predict(x, verbose=0)[0]
    class_id = int(np.argmax(proba))
    return class_id, float(proba[class_id]), proba


def detect_and_classify_faces(image_bgr, classifier=None, det_conf=0.35, imgsz=640):
    if classifier is None:
        classifier = final_model

    detections = []
    for box, yolo_conf in detect_faces_yolo(image_bgr, conf=det_conf, imgsz=imgsz):
        face_crop, expanded_box = crop_face_with_margin(image_bgr, box, margin=0.12)
        if face_crop is None or face_crop.size == 0:
            continue

        class_id, emotion_conf, proba = classify_face_crop(face_crop, classifier)
        detections.append({
            "box": tuple(map(int, expanded_box)),
            "yolo_conf": yolo_conf,
            "class_id": class_id,
            "emotion": CLASS_NAMES_FR[class_id],
            "emotion_conf": emotion_conf,
            "probabilities": proba,
        })
    return detections


def draw_face_detections(image_bgr, detections):
    out = image_bgr.copy()
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        label = f'{det["emotion"]} {det["emotion_conf"]:.2f} | face {det["yolo_conf"]:.2f}'
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)

        text_y = max(18, y1 - 7)
        cv2.putText(
            out, label, (x1, text_y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA
        )
    return out


def analyze_image_file(image_path, classifier=None, det_conf=0.35, imgsz=640, figsize=(12, 8)):
    image_path = str(image_path)
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"Image illisible : {image_path}")

    detections = detect_and_classify_faces(
        image_bgr, classifier=classifier, det_conf=det_conf, imgsz=imgsz
    )
    annotated = draw_face_detections(image_bgr, detections)

    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{len(detections)} visage(s) détecté(s) et analysé(s)")
    plt.show()

    if detections:
        summary = pd.DataFrame([
            {
                "Visage": i + 1,
                "Expression": d["emotion"],
                "Confiance expression": d["emotion_conf"],
                "Confiance YOLO": d["yolo_conf"],
                "Bounding box": d["box"],
            }
            for i, d in enumerate(detections)
        ])
        display(summary.style.format({
            "Confiance expression": "{:.3f}",
            "Confiance YOLO": "{:.3f}",
        }))
    else:
        print("Aucun visage détecté. Essayer une image plus nette ou diminuer det_conf.")

    return annotated, detections

### 8.2 Démonstration sur une image contenant plusieurs personnes

Définissez `DEMO_IMAGE_PATH` avec une photo de groupe. La cellule est volontairement non bloquante : si aucun fichier n'est fourni, le notebook continue sans erreur.

Dans Google Colab, vous pouvez téléverser une image avec l'icône **Files** puis indiquer par exemple `"/content/groupe.jpg"`.

In [ ]:
DEMO_IMAGE_PATH = Path("content/image.jpg") 

print("Dossier courant :", Path.cwd())
print("Image :", DEMO_IMAGE_PATH.resolve())
print("Existe :", DEMO_IMAGE_PATH.exists())

if DEMO_IMAGE_PATH and Path(DEMO_IMAGE_PATH).exists():
    demo_image_annotated, demo_image_detections = analyze_image_file(
        DEMO_IMAGE_PATH,
        classifier=final_model,
        det_conf=0.35,
        imgsz=640,
    )
else:
    print("Démo image prête. Renseigner DEMO_IMAGE_PATH pour l'exécuter sur une photo de groupe.")

# 9. Extension bonus — Analyse d'une vidéo

Le pipeline vidéo applique le même traitement aux images successives :

**Vidéo → frames → YOLO-Face → crops 48×48 gris → CNN FER2013 → expression + score → vidéo annotée**

Le paramètre `detect_every` permet de détecter tous les `N` frames. Avec `1`, chaque frame est analysée ; une valeur `2` ou `3` accélère la démonstration et réutilise brièvement les dernières boîtes entre deux détections.

In [ ]:
from collections import Counter


def process_video(
    input_path,
    output_path="fer2013_video_analysee.mp4",
    classifier=None,
    det_conf=0.35,
    imgsz=640,
    detect_every=1,
    max_frames=None,
):
    if classifier is None:
        classifier = final_model

    input_path = str(input_path)
    output_path = Path(output_path)
    raw_output = output_path.with_name(output_path.stem + "_raw.mp4")

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Vidéo impossible à ouvrir : {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        str(raw_output),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError("Impossible de créer la vidéo de sortie.")

    frame_idx = 0
    cached_detections = []
    emotion_counter = Counter()
    face_instances = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if max_frames is not None and frame_idx >= max_frames:
            break

        if frame_idx % max(1, int(detect_every)) == 0:
            cached_detections = detect_and_classify_faces(
                frame,
                classifier=classifier,
                det_conf=det_conf,
                imgsz=imgsz,
            )
            for det in cached_detections:
                emotion_counter[det["emotion"]] += 1
            face_instances += len(cached_detections)

        annotated = draw_face_detections(frame, cached_detections)
        writer.write(annotated)
        frame_idx += 1

    cap.release()
    writer.release()

    # Conversion H.264 pour une lecture plus fiable dans les navigateurs/Colab.
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg:
        cmd = [
            ffmpeg, "-y", "-loglevel", "error",
            "-i", str(raw_output),
            "-vcodec", "libx264", "-pix_fmt", "yuv420p",
            str(output_path),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0 and output_path.exists():
            raw_output.unlink(missing_ok=True)
        else:
            raw_output.replace(output_path)
    else:
        raw_output.replace(output_path)

    duration_s = frame_idx / fps if fps > 0 else np.nan
    stats = {
        "frames": frame_idx,
        "fps": fps,
        "duration_s": duration_s,
        "face_instances_on_detection_frames": face_instances,
        "emotion_counts": dict(emotion_counter),
        "output_path": str(output_path),
    }
    return output_path, stats

### 9.1 Démonstration sur une courte vidéo

Renseignez le chemin d'une vidéo courte. Pour une soutenance fluide, une séquence de quelques secondes contenant une ou plusieurs personnes suffit. Le résultat est sauvegardé en MP4 et affichable directement dans Colab.

In [ ]:
DEMO_VIDEO_PATH = Path("content/video.mp4")  # Exemple : "/content/demo_visages.mp4"

if DEMO_VIDEO_PATH and Path(DEMO_VIDEO_PATH).exists():
    video_output, video_stats = process_video(
        DEMO_VIDEO_PATH,
        output_path="fer2013_video_analysee.mp4",
        classifier=final_model,
        det_conf=0.35,
        imgsz=640,
        detect_every=1,
    )

    print("Statistiques :", video_stats)

    try:
        from IPython.display import Video, display
        display(Video(str(video_output), embed=True))
    except Exception:
        print("Vidéo générée :", video_output)
else:
    print("Démo vidéo prête. Renseigner DEMO_VIDEO_PATH pour générer la vidéo annotée.")

# Conclusion générale — Parties 1 à 9

Le projet couvre maintenant l'ensemble de la progression demandée :

1. **Préparation de FER2013** : contrôle des données, labels, normalisation et splits train/validation/test.
2. **Baseline dense** : point de référence sans convolution.
3. **CNN Keras** : extraction de caractéristiques par convolution/pooling et classification Softmax.
4. **Entraînement** : choix de la loss, Adam, batch size, callbacks et courbes d'apprentissage.
5. **Évaluation** : accuracy, macro-F1, matrice de confusion, analyse des classes et erreurs individuelles.
6. **Expérimentations** : Data Augmentation, Dropout et learning rate, avec comparaison sur validation.
7. **Enrichissement** : Transfer Learning MobileNetV2 et fine-tuning contrôlé.
8. **Multi-visages** : YOLO-Face localise chaque visage, puis le CNN FER2013 prédit son expression.
9. **Vidéo** : le pipeline complet est appliqué aux frames successives et produit une vidéo annotée.

## Points à montrer pendant la soutenance

- le tableau de distribution des classes et quelques exemples FER2013 ;
- l'architecture et le nombre de paramètres du CNN ;
- les courbes `loss/accuracy` et le diagnostic de surapprentissage ;
- la matrice de confusion et deux ou trois erreurs intéressantes ;
- le tableau des trois expériences de la partie 6 ;
- la comparaison validation du Transfer Learning ;
- une **photo de groupe** avec plusieurs bounding boxes + expressions ;
- une **courte vidéo** générée par la partie 9.

Le détecteur et le classifieur ont des rôles distincts : **YOLO répond “où est le visage ?”**, tandis que **le CNN FER2013 répond “quelle expression ?”**.

## Checklist finale

- [x] Partie 1 — Recherche, compréhension et préparation des données
- [x] Partie 2 — Modèle de référence dense
- [x] Partie 3 — CNN avec Keras
- [x] Partie 4 — Entraînement et courbes
- [x] Partie 5 — Évaluation, matrice de confusion et analyse des erreurs
- [x] Partie 6 — Au moins 3 expériences + choix du modèle
- [x] Partie 7 — Transfer Learning / fine-tuning
- [x] Partie 8 — Détection de plusieurs visages avec YOLO-Face + classification FER2013
- [x] Partie 9 — Traitement d'une vidéo et génération d'une sortie annotée
- [ ] À exécuter avant la soutenance : renseigner `DEMO_IMAGE_PATH` et `DEMO_VIDEO_PATH` avec vos propres fichiers de démonstration